# 04 - Next-day recovery prediction

## Project goal

This notebook evaluates whether aggregate Apple Watch features contain predictive signal for a personal recovery trend one day ahead. It compares machine-learning models with persistence and 7-day rolling mean baselines. The analysis describes association in one personal time series and is **not a clinical diagnosis**.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import Image, display

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.plot_ml_results import (
    extract_feature_importance,
    generate_all_figures,
    load_plot_inputs,
)

PREDICTIONS_PATH = PROJECT_ROOT / "reports" / "ml_predictions.csv"
REGRESSION_METRICS_PATH = PROJECT_ROOT / "reports" / "ml_regression_metrics.csv"
CLASSIFICATION_METRICS_PATH = PROJECT_ROOT / "reports" / "ml_classification_metrics.csv"
ML_DATASET_PATH = PROJECT_ROOT / "data" / "processed" / "ml_recovery_dataset.csv"
FIGURE_DIR = PROJECT_ROOT / "reports" / "figures"

test_predictions, regression_metrics, classification_metrics, ml_dataset = load_plot_inputs(
    PREDICTIONS_PATH,
    REGRESSION_METRICS_PATH,
    CLASSIFICATION_METRICS_PATH,
    ML_DATASET_PATH,
)
generate_all_figures(
    test_predictions,
    regression_metrics,
    ml_dataset,
    FIGURE_DIR,
)

## Dataset overview

The ML dataset contains daily aggregate features, historical lag/rolling/trend features, missingness indicators, calendar features, and next-day targets. Rows with missing targets are not used for model evaluation; feature missingness is handled inside sklearn pipelines.

In [ ]:
dataset_overview = pd.Series({
    "dataset_rows": len(ml_dataset),
    "dataset_columns": len(ml_dataset.columns),
    "date_start": ml_dataset["date"].min().date(),
    "date_end": ml_dataset["date"].max().date(),
    "valid_regression_targets": ml_dataset["target_recovery_next_day"].notna().sum(),
    "missing_regression_targets": ml_dataset["target_recovery_next_day"].isna().sum(),
    "test_rows": len(test_predictions),
})
dataset_overview.to_frame("value")

## Train/test split explanation

The training module uses chronological splitting rather than random sampling. This dataset covers the fixed window, so training uses 2025-06-01 through 2025-10-31 and testing uses 2025-11-01 through 2025-12-31. Every training date precedes every test date.

In [ ]:
all_predictions = pd.read_csv(PREDICTIONS_PATH, parse_dates=["date"])
split_summary = all_predictions.groupby("split").agg(
    rows=("date", "size"),
    start_date=("date", "min"),
    end_date=("date", "max"),
)
split_summary

## Baseline comparison

The key question is whether the best ML regressor consistently outperforms both simple baselines. Because persistence and rolling predictions can be missing, the `common_*` columns compare every method on the same test dates.

In [ ]:
baseline_columns = [
    "model",
    "model_type",
    "MAE",
    "RMSE",
    "R2",
    "evaluation_samples",
    "common_MAE",
    "common_RMSE",
    "common_R2",
    "common_evaluation_samples",
]
regression_metrics[baseline_columns].sort_values("common_MAE")

## Regression results

Predicted-vs-actual distance summarizes score error, the timeline shows when predictions diverge, and residuals identify the largest misses. All three views below use only the test set.

In [ ]:
display(regression_metrics.sort_values(["model_type", "MAE"]))

for filename in (
    "ml_predicted_vs_actual.png",
    "ml_recovery_prediction_timeline.png",
    "ml_residuals.png",
):
    display(Image(filename=FIGURE_DIR / filename))

## Classification results

Low recovery is defined as a next-day recovery score below 40. Accuracy alone is not sufficient because the test set contains few positive examples; precision, recall, F1, ROC-AUC, and the confusion matrix provide the necessary context.

In [ ]:
display(classification_metrics.sort_values("F1", ascending=False))

display(Image(filename=FIGURE_DIR / "ml_confusion_matrix.png"))

## Feature importance

Native Random Forest importance ranks features used by the reported best regression model. Importance can reveal predictive signal but does not establish a causal relationship.

In [ ]:
importance, best_regression_model, importance_measure = extract_feature_importance(
    ml_dataset,
    regression_metrics,
)
print(f"Best regression model: {best_regression_model}")
print(f"Importance measure: {importance_measure}")
if importance is not None:
    display(importance.head(15))

display(Image(filename=FIGURE_DIR / "ml_feature_importance.png"))

## Limitations

- The test set has only 26 target-valid dates and only 3 low-recovery examples.
- Missing health observations reduce baseline coverage and require model-side imputation.
- The target is derived from the existing rule-based personal recovery trend, so the model learns that definition rather than an independent outcome.
- Feature importance is model-specific and should be interpreted as association, not causation.
- Model selection used the same test period reported here; a later untouched time window is needed to assess stability.
- Results come from one personal time series and are not a clinical diagnosis.

## Conclusion

The Random Forest regressor improves on persistence, but it does not consistently outperform the 7-day rolling mean baseline on common test dates. Regression R2 remains below zero, and classification performance is unstable with only three positive test examples. The current features show limited association with next-day recovery prediction, but they do not provide stable incremental predictive signal beyond a simple rolling baseline. The ML extension therefore supports a conservative project conclusion: more target-valid observations, stronger feature coverage, and an untouched future evaluation window are required before treating the model as reliable.